# 11 — TF/TPU Hazırlık Oturumu (ÜCRETSİZ CPU çalışma zamanı, İDEMPOTENT)

**Runtime: CPU seçin** — kredi harcamaz. Her hücre korumalıdır: daha önce
üretilmiş çıktı Drive'da varsa hücre kendini atlar. Oturum çökerse tek yapman
gereken **yeniden bağlanıp "Tümünü çalıştır"** demektir — kaldığı yerden sürer.

Train dönüştürmesi oturum ömrüne sığmadığı için İKİ PARÇAYA bölünmüştür
(P1: satır 0-3209, P2: 3209-son). Her parça ~3.5-4 saat sürer ve kendi
işaret dosyasıyla (.train_pX_done) tamamlandığını kaydeder.

Bittiğinde `10_tf_tpu_training.ipynb` TPU oturumunda çalıştırılabilir
(glob'lar p1/p2 shard'larını otomatik yakalar, notebook 10 değişmedi).

In [ ]:
# Drive bağla
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/AFETSONAR")
assert DRIVE.exists(), "AFETSONAR klasörü bulunamadı"

In [ ]:
# tf-port branch'ini klonla ve paketi kur (~3 dk)
import os
if not os.path.exists("/content/calamitas-ai"):
    !git clone -b tf-port https://github.com/Runc-Dev/calamitas-ai.git /content/calamitas-ai
else:
    !cd /content/calamitas-ai && git pull
%cd /content/calamitas-ai
!pip install -q -e .
!pip install -q "transformers==5.7.0"  # checkpoint'ların doğrulandığı sürüm

In [ ]:
# 1) Ağırlık export'u — Drive'da npz varsa atlanır
import os
CKPT = "/content/calamitas-ai/checkpoints/teacher_v4_best_ema.pth"
if os.path.exists(f"{DRIVE}/tf_export/teacher_v4_ema_hf.npz"):
    print("export ATLANDI — npz'ler zaten Drive tf_export/ içinde")
else:
    !mkdir -p /content/calamitas-ai/checkpoints
    if not os.path.exists(CKPT):
        !cp "{DRIVE}/checkpoints/teacher/teacher_v4_best_ema.pth" "{CKPT}"
    !python scripts/export_weights_npz.py --checkpoint "{CKPT}"
    !mkdir -p "{DRIVE}/tf_export"
    !cp checkpoints/export/*.npz checkpoints/export/manifest.json "{DRIVE}/tf_export/"
    !cp tests_tf/golden/loss_inputs.npz tests_tf/golden/loss_values.json "{DRIVE}/tf_export/" 2>/dev/null || true
    !ls -lh "{DRIVE}/tf_export/"

In [ ]:
# 2) GATE torch referansı — gate_ref.json Drive'da varsa atlanır (40 dk kazanç)
import os, json
GATE_REF = f"{DRIVE}/tf_export/gate_ref.json"
if os.path.exists(GATE_REF):
    print("gate referansı ATLANDI — mIoU_no_bg =",
          json.load(open(GATE_REF))["metrics"]["miou_no_bg"])
else:
    import pandas as pd
    df = pd.read_csv(DRIVE / "data/splits/test_v3.csv").head(200)
    df.to_csv("/content/test200.csv", index=False)
    df.to_csv(f"{DRIVE}/tf_export/test200.csv", index=False)
    print(f"test200.csv: {len(df)} satır")
    if not os.path.exists(CKPT):
        !cp "{DRIVE}/checkpoints/teacher/teacher_v4_best_ema.pth" "{CKPT}"
    !python scripts/evaluate.py         --model "{CKPT}"         --test-csv /content/test200.csv         --batch-size 2 --device cpu         --output /content/gate_ref.json
    !cp /content/gate_ref.json "{DRIVE}/tf_export/"
    print(json.load(open("/content/gate_ref.json"))["metrics"]["miou_no_bg"])

In [ ]:
# 3) GATE + VAL shard'ları — mevcutsa atlanır (test200.csv artık Drive'dan okunur)
import glob as _g
!mkdir -p "{DRIVE}/tfrecords"
if _g.glob(f"{DRIVE}/tfrecords/gate_dmg-*"):
    print("gate shard'ları ATLANDI")
else:
    !python scripts_tf/convert_to_tfrecords.py         --csv "{DRIVE}/tf_export/test200.csv" --split gate         --out-dir "{DRIVE}/tfrecords"
if _g.glob(f"{DRIVE}/tfrecords/val_nodmg-*"):
    print("val shard'ları ATLANDI")
else:
    !python scripts_tf/convert_to_tfrecords.py         --csv "{DRIVE}/data/splits/val_v3.csv" --split val         --out-dir "{DRIVE}/tfrecords"

In [ ]:
# 4) Eski (bölünmemiş) ölü koşulardan kalan KISMİ train shard'larını temizle.
# Kalırlarsa notebook 10'un train_dmg-* glob'u kesik dosyaları da toplar.
# [0-9] kalıbı yalnızca eski adlandırmayı yakalar; p1/p2 dosyalarına dokunmaz.
import glob as _g, os
old = (_g.glob(f"{DRIVE}/tfrecords/train_dmg-[0-9]*.tfrecord")
       + _g.glob(f"{DRIVE}/tfrecords/train_nodmg-[0-9]*.tfrecord")
       + _g.glob(f"{DRIVE}/tfrecords/train_cp-[0-9]*.tfrecord"))
for f in old:
    os.remove(f)
print(f"temizlendi: {len(old)} eski kısmi shard")

In [ ]:
# 5) TRAIN P1 — satır 0:3209 (+Copy-Paste), ~3.5-4 saat. Bitince işaret dosyası.
import os
if os.path.exists(f"{DRIVE}/tfrecords/.train_p1_done"):
    print("P1 ATLANDI — işaret dosyası var")
else:
    !python scripts_tf/convert_to_tfrecords.py         --csv "{DRIVE}/data/splits/train_v3.csv" --split train         --copy-paste --rows 0:3209 --shard-suffix p1         --out-dir "{DRIVE}/tfrecords" && touch "{DRIVE}/tfrecords/.train_p1_done"
    assert os.path.exists(f"{DRIVE}/tfrecords/.train_p1_done"),         "P1 tamamlanamadı — hücreyi yeniden çalıştırın"

In [ ]:
# 6) TRAIN P2 — satır 3209:son (+Copy-Paste), ~3.5-4 saat.
import os
if os.path.exists(f"{DRIVE}/tfrecords/.train_p2_done"):
    print("P2 ATLANDI — işaret dosyası var")
else:
    !python scripts_tf/convert_to_tfrecords.py         --csv "{DRIVE}/data/splits/train_v3.csv" --split train         --copy-paste --rows 3209: --shard-suffix p2         --out-dir "{DRIVE}/tfrecords" && touch "{DRIVE}/tfrecords/.train_p2_done"
    assert os.path.exists(f"{DRIVE}/tfrecords/.train_p2_done"),         "P2 tamamlanamadı — hücreyi yeniden çalıştırın"

In [ ]:
# 7) Özet
import glob as _g, os
fam = lambda p: len(_g.glob(f"{DRIVE}/tfrecords/{p}"))
print(f"train_dmg-p1: {fam('train_dmg-p1-*')}  train_dmg-p2: {fam('train_dmg-p2-*')}")
print(f"train_nodmg p1/p2: {fam('train_nodmg-p1-*')}/{fam('train_nodmg-p2-*')}  "
      f"train_cp p1/p2: {fam('train_cp-p1-*')}/{fam('train_cp-p2-*')}")
print(f"val: {fam('val_*')}  gate: {fam('gate_*')}  eski kalıntı: {fam('train_dmg-[0-9]*')}")
!du -sh "{DRIVE}/tfrecords"
done = (os.path.exists(f"{DRIVE}/tfrecords/.train_p1_done")
        and os.path.exists(f"{DRIVE}/tfrecords/.train_p2_done"))
print("HAZIRLIK TAMAM — 10_tf_tpu_training.ipynb TPU oturumunda çalıştırılabilir"
      if done else "HAZIRLIK EKSİK — eksik parçayı yeniden çalıştırın")